In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# LaB₆ — neutron powder, constant wavelength, sample absorption

In [2]:
import easydiffraction as ed
from easydiffraction import ExperimentFactory
from easydiffraction import StructureFactory
from easydiffraction.analysis import verification as verify

## Build the project

In [3]:
project = ed.Project()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Define the structure

In [4]:
structure = StructureFactory.from_scratch(name='lab6')
structure.space_group.name_h_m = 'P m -3 m'  # FullProf Space group symbol
structure.cell.length_a = 4.156885  # FullProf a
structure.atom_sites.create(
    label='La',  # FullProf Atom
    type_symbol='La',  # FullProf Typ
    fract_x=0.0,  # FullProf X
    fract_y=0.0,  # FullProf Y
    fract_z=0.0,  # FullProf Z
    adp_type='Biso',  # FullProf Biso
    adp_iso=0.59951,  # FullProf Biso
)
structure.atom_sites.create(
    label='B',  # FullProf Atom
    type_symbol='11B',  # FullProf "B11"
    fract_x=0.19978,  # FullProf X
    fract_y=0.5,  # FullProf Y
    fract_z=0.5,  # FullProf Z
    adp_type='Biso',  # FullProf Biso
    adp_iso=0.44499,  # FullProf Biso
)

project.structures.add(structure)

## Load the FullProf reference

In [5]:
FULLPROF_PROJECT_DIR = 'pd-neut-cwl_tch-fcj_lab6'
FULLPROF_PRF_FILE = 'ECH0030684_LaB6_1p622A.prf'
FULLPROF_BAC_FILE = 'ECH0030684_LaB6_1p622A.bac'
FULLPROF_ZERO = -0.21110  # FullProf Zero
FULLPROF_SCALE = 141.1285  # FullProf Scale
FULLPROF_WAVELENGTH = 1.622527  # FullProf Lambda
FULLPROF_U = 0.089664  # FullProf U
FULLPROF_V = -0.375792  # FullProf V
FULLPROF_W = 0.476524  # FullProf W
FULLPROF_X = 0.0  # FullProf X
FULLPROF_Y = 0.052425  # FullProf Y
FULLPROF_SYCOS = 0.05281  # FullProf SyCos
FULLPROF_SYSIN = 0.09068  # FullProf SySin
FULLPROF_S_L = 0.08000  # FullProf S_L
FULLPROF_D_L = 0.08000  # FullProf D_L

x, calc_fullprof = verify.load_fullprof_calc_profile(
    FULLPROF_PROJECT_DIR,
    FULLPROF_PRF_FILE,
    FULLPROF_BAC_FILE,
    FULLPROF_ZERO,
)

## Create the experiment

In [6]:
experiment = ExperimentFactory.from_scratch(
    name='lab6',
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
    scattering_type='bragg',
)
verify.set_reference_as_measured(experiment, x, calc_fullprof)

experiment.linked_phases.create(id='lab6', scale=FULLPROF_SCALE)

experiment.instrument.setup_wavelength = FULLPROF_WAVELENGTH
experiment.instrument.calib_twotheta_offset = FULLPROF_ZERO

experiment.peak.broad_gauss_u = FULLPROF_U
experiment.peak.broad_gauss_v = FULLPROF_V
experiment.peak.broad_gauss_w = FULLPROF_W
experiment.peak.broad_lorentz_x = FULLPROF_X
experiment.peak.broad_lorentz_y = FULLPROF_Y
# Engine-specific corrections are applied in each engine's section below:
# SyCos/SySin (cryspy only) and the FCJ S_L/D_L asymmetry (crysfml only).
# Sample absorption (muR = 0.7) is modelled by neither engine.

project.experiments.add(experiment)

## ed-cryspy VS FullProf

In [7]:
experiment.instrument.calib_sample_displacement = FULLPROF_SYCOS
experiment.instrument.calib_sample_transparency = FULLPROF_SYSIN

experiment.calculator.type = 'cryspy'
project.analysis.calculate()
calc_ed_cryspy = experiment.data.intensity_calc

project.display.pattern_comparison(
    'lab6',
    reference=calc_fullprof,
    candidate=calc_ed_cryspy,
    reference_label='FullProf',
    candidate_label='ed-cryspy',
)

Calculator for experiment 'lab6' already set to


cryspy


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Fit ed-cryspy to FullProf

In [8]:
experiment.linked_phases['lab6'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True
experiment.instrument.calib_sample_displacement.free = True
experiment.instrument.calib_sample_transparency.free = True

project.analysis.fit()
project.display.fit.results()

project.analysis.calculate()
calc_ed_cryspy_refined = experiment.data.intensity_calc

project.display.pattern_comparison(
    'lab6',
    reference=calc_fullprof,
    candidate=calc_ed_cryspy_refined,
    reference_label='FullProf',
    candidate_label='ed-cryspy (refined)',
)

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'lab6' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ed-crysfml VS FullProf

In [9]:
experiment.calculator.type = 'crysfml'
experiment.peak.type = 'thompson-cox-hastings'
experiment.linked_phases['lab6'].scale = FULLPROF_SCALE
experiment.linked_phases['lab6'].scale.free = False
experiment.instrument.calib_twotheta_offset = FULLPROF_ZERO
experiment.instrument.calib_twotheta_offset.free = False
experiment.instrument.calib_sample_displacement.free = False
experiment.instrument.calib_sample_transparency.free = False
experiment.peak.broad_gauss_u = FULLPROF_U
experiment.peak.broad_gauss_v = FULLPROF_V
experiment.peak.broad_gauss_w = FULLPROF_W
experiment.peak.broad_lorentz_x = FULLPROF_X
experiment.peak.broad_lorentz_y = FULLPROF_Y
experiment.peak.asym_fcj_1 = FULLPROF_S_L
experiment.peak.asym_fcj_2 = FULLPROF_D_L

project.analysis.calculate()
calc_ed_crysfml = experiment.data.intensity_calc

project.display.pattern_comparison(
    'lab6',
    reference=calc_fullprof,
    candidate=calc_ed_crysfml,
    reference_label='FullProf',
    candidate_label='ed-crysfml',
)

Calculator for experiment 'lab6' changed to


crysfml


⚠️ Switching peak profile type adds these settings with defaults:                                                                 
   • asym_fcj_1=0.0                                                                                                               
   • asym_fcj_2=0.0                                                                                                               


⚠️ Switching peak profile type resets these settings to defaults:                                                                 
   • broad_gauss_u: 0.089664 -> 0.01                                                                                              
   • broad_gauss_v: -0.375792 -> -0.01                                                                                            
   • broad_gauss_w: 0.476524 -> 0.02                                                                                              
   • broad_lorentz_y: 0.052425 -> 0.0                                                                                             


Peak profile type for experiment 'lab6' changed to


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Fit ed-crysfml to FullProf

In [10]:
experiment.linked_phases['lab6'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True

project.analysis.fit()
project.display.fit.results()

project.analysis.calculate()
calc_ed_crysfml_refined = experiment.data.intensity_calc

project.display.pattern_comparison(
    'lab6',
    reference=calc_fullprof,
    candidate=calc_ed_crysfml_refined,
    reference_label='FullProf',
    candidate_label='ed-crysfml (refined)',
)

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'lab6' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Agreement check

In [11]:
verify.assert_patterns_agree(
    [
        ('cryspy vs FullProf', calc_fullprof, calc_ed_cryspy),
        ('crysfml vs FullProf', calc_fullprof, calc_ed_crysfml),
    ],
    raise_on_failure=False,
)

,Comparison,Metric,Expected,Actual,OK
1,cryspy vs FullProf,Profile diff (%),< 3,202.68,❌
2,,Max deviation (%),< 5,208.04,❌
3,,Area ratio,0.98 to 1.02,2.8885,❌
4,,Shape correlation,> 0.99,0.9719,❌
5,crysfml vs FullProf,Profile diff (%),< 3,112.91,❌
6,,Max deviation (%),< 5,117.65,❌
7,,Area ratio,0.98 to 1.02,1.9951,❌
8,,Shape correlation,> 0.99,0.9541,❌


False